In [0]:
from pyspark.sql import functions as F

posts = spark.table("bright_data_pinterest_profiles_and_posts_datasets.datasets.pinterest_posts")

df = (
    posts
    .withColumn("year", F.year("date_posted"))
    .withColumn("hashtags_raw", F.col("hashtags"))
    .withColumn(
        "hashtags_raw",
        F.when(F.col("hashtags_raw").isin("null", "", "[]"), F.lit(None)).otherwise(F.col("hashtags_raw"))
    )
)

# split logic depends on the column format — start simple:
tags = (
    df
    .where(F.col("hashtags_raw").isNotNull())
    .withColumn("tag", F.explode(F.split(F.col("hashtags_raw"), r"[,]+")))
    #.withColumn("tag", F.regexp_replace("tag", r"[^a-z0-9_#-]", ""))  # keep simple tokens
    #.withColumn("tag", F.regexp_replace("tag", r"^#+", ""))          # strip leading ###
    .where(F.length("tag") > 1)
)

tag_counts = (
    tags
    .groupBy("year", "tag")
    .agg(F.count("*").alias("cnt"))
    .orderBy("year", F.desc("cnt"))
)
tag_counts = tag_counts.where(F.col("year").isNotNull())
tag_counts.createOrReplaceTempView("tag_counts")
display(tag_counts)

In [0]:
%sql
SELECT year(p.date_posted) AS year,
  trim(tag) AS tag,
  count(*) AS cnt
FROM pinterest_posts p
LATERAL VIEW explode(split(p.hashtags, ',')) AS tag
WHERE tag IS NOT NULL 
AND p.date_posted IS NOT NULL 
AND length(trim(tag)) > 1
GROUP BY year, tag
order by cnt desc, year asc, tag desc

In [0]:
%sql 
describe pinterest_posts;

In [0]:
%sql
SELECT
  year(p.date_posted) AS year,
  trim(tag) AS tag,
  count(*) AS cnt
FROM pinterest_posts p
CROSS JOIN lateral ( select explode(split(p.hashtags, ',')) AS tag ) x
WHERE x.tag IS NOT NULL
  AND p.date_posted IS NOT NULL
  AND length(trim(tag)) > 1
GROUP BY year, tag
ORDER BY cnt DESC, year ASC, tag DESC;

In [0]:
%pip install wordcloud

from wordcloud import WordCloud
import matplotlib.pyplot as plt

# or tag_clouds
years = [r["year"] for r in _sqldf.select("year").distinct().orderBy("year").collect()]

for y in years:
    top = (
        _sqldf # or tag_counts
        .where(F.col("year") == y)
        .orderBy(F.desc("cnt"))
        .limit(200)
        .toPandas()
    )
    freqs = dict(zip(top["tag"], top["cnt"]))
    wc = WordCloud(width=900, height=450, background_color="white").generate_from_frequencies(freqs)

    plt.figure()
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Top tags — {y}")
    plt.show()